# This code generates the relationsship figures between position and price and also poistion and CCP

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
Path.cwd()
DATA = Path('train.parquet')
RES = Path('res')
RES.mkdir(exist_ok=True)

MAX_POS = 25     # the last positions are very noisy (as in getting_started)
SPECIAL_SLOTS = [5, 11, 17, 23]   # special slots: empty in most sessions, so very few observations

# Import data

In [ ]:
cols = ['srch_id', 'position', 'random_bool', 'click_bool', 'booking_bool', 'price_usd']
dat = pd.read_parquet(DATA, columns=cols)

# as in getting_started: a few listings have a price of zero, so log price fails: drop their sessions
price_min = dat.groupby('srch_id')['price_usd'].transform('min')
dat = dat[price_min > 0]

dat['log_price'] = np.log(dat['price_usd'])
# price relative to the session's average: compares hotels within the same list
dat['log_price_rel'] = dat['log_price'] - dat.groupby('srch_id')['log_price'].transform('mean')
dat['ordering'] = dat['random_bool'].map({0: 'Expedia ranking', 1: 'Random ranking'})
dat = dat.query('position <= @MAX_POS')

print(f"Sessions: {dat['srch_id'].nunique():,}  Listings: {len(dat):,}")

# Helper functions

In [ ]:
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Computer Modern Roman', 'CMU Serif', 'Times New Roman', 'DejaVu Serif'],
    'mathtext.fontset': 'cm',
    'font.size': 10, 'axes.labelsize': 10, 'xtick.labelsize': 9, 'ytick.labelsize': 9, 'legend.fontsize': 9,
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.linewidth': 0.8,
    'xtick.major.width': 0.8, 'ytick.major.width': 0.8,
    'axes.grid': True, 'axes.grid.axis': 'y', 'grid.color': '#D7D7E9', 'grid.linewidth': 0.6,
    'savefig.bbox': 'tight', 'savefig.pad_inches': 0.02,
})

# palette: #2D004B, #8170AC, #D7D7E9, #FDDDB3, #DD841F, #7F3B08
# different markers and line styles, so the figures also read in black and white
STYLE = {'Expedia ranking': dict(color='#2D004B', marker='o', ls='-'),
         'Random ranking':  dict(color='#DD841F', marker='s', ls='--')}

from matplotlib.lines import Line2D

def plot_by_position(df, y, ylabel, fname, hline=None):
    '''Mean of y by position for each ordering; saves pdf in RES and returns the table.
    Lines connect only the regular positions; the special slots are shown as hollow,
    unconnected markers, and the y-axis is scaled from the regular positions only.'''
    tab = df.groupby(['ordering', 'position'])[y].mean().unstack('ordering')
    special = tab.index.isin(SPECIAL_SLOTS)
    reg, spec = tab[~special], tab[special]

    fig, ax = plt.subplots(figsize=(6.0, 3.4))
    if hline is not None:
        ax.axhline(hline, color='0.4', lw=0.8, zorder=1)
    for ordering in ['Expedia ranking', 'Random ranking']:
        ax.plot(reg.index, reg[ordering], ms=4, lw=1.4, label=ordering, zorder=3, **STYLE[ordering])

    # freeze the y-axis at the range of the regular positions before adding the special slots
    ax.set_ylim(ax.get_ylim())
    for ordering in ['Expedia ranking', 'Random ranking']:
        st = STYLE[ordering]
        ax.plot(spec.index, spec[ordering], ls='none', marker=st['marker'], ms=4,
                mec=st['color'], mfc='none', mew=1.0, zorder=3)

    ax.set_xlabel('Position in list')
    ax.set_ylabel(ylabel)
    ax.set_xticks(range(1, MAX_POS + 1, 2))
    ax.set_xlim(0.5, MAX_POS + 0.5)
    handles, labels = ax.get_legend_handles_labels()
    handles.append(Line2D([], [], ls='none', marker='o', ms=4, mec='0.5', mfc='none', mew=1.0))
    labels.append(f"Special slots ({', '.join(map(str, SPECIAL_SLOTS))})")
    ax.legend(handles, labels, frameon=False)
    fig.savefig(RES / f'{fname}.pdf')
    plt.show()
    return tab

# Figure 1.1a

In [ ]:
tab_a = plot_by_position(dat, 'log_price_rel', 'Log price relative to session mean',
                         'fig_1_1a_price_by_position', hline=0)

# Figure 1.1b

In [ ]:
tab_b = plot_by_position(dat.query('click_bool == 1'), 'booking_bool',
                         r'$\Pr(\mathrm{book} \mid \mathrm{click})$', 'fig_1_1b_book_given_click')

# Save

In [ ]:
sorted(p.name for p in RES.iterdir())